# 06 - Federated Simulation
Runs FedAvg across regional shards and compares with centralized training metrics.

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from falabella_risk.federated.federated_training import train_federated

report = train_federated(
    features_path=Path('data/processed/features.parquet'),
    model_out=Path('models/federated_model.pt'),
    report_out=Path('reports/federated_report.json'),
    seed=42,
    rounds=8,
    local_epochs=2,
    learning_rate=0.01,
)
report

In [ ]:
loaded = json.loads(Path('reports/federated_report.json').read_text(encoding='utf-8'))
pd.DataFrame([
    {'mode': 'federated', **loaded['federated_metrics']},
    {'mode': 'centralized', **loaded['centralized_metrics']},
])

In [ ]:
print('AUC gap vs centralized:', round(loaded['auc_gap_vs_centralized'], 4))
print('Within 2-point target:', abs(loaded['auc_gap_vs_centralized']) <= 0.02)